# Draft: the model in the style of `archive/williams/`

Plain functions and named constants, in place of one class and a parameter dictionary, with the model, the timestep, the simulation and the controller kept apart. The sections and their order follow `RobotSimulationOpenLoopModel_HenryConstants.ipynb`.

## 🤖 Complete Plant Simulation Specification: Discrete-Time Open-Loop Asymmetric 4-Omni Robot Model

This cell specifies the comprehensive mathematical framework for the 4-wheel omni-directional mobile robot simulation. The architecture accounts for **asymmetric wheel layouts**, **dynamic multi-axis weight transfer (pitch and roll)**, **non-calibrated hardware parameters**, and **LuGre bristle floor friction boundaries** evaluated via **implicit (backward Euler) discrete-time updates** at a fixed sampling interval $\Delta T = 1\text{ ms}$ ($1\text{ kHz}$ loop rate).

---
<details>
<summary><b> Open this for the full specification of the simulation equations</b></summary>


### 1. Parameters & Variable Definitions

#### 🛠️ Hardware & Structural Parameters (Henry Calibrated Constants)
* $M_r$ : Total mass of the robot
* $J_{zz}$: Rotational mass moment of inertia around vertical Z-axis
* $J_w $: Rotational moment of inertia of a wheel/rotor assembly
* $R_b$ : Geometric base radius from the base centre to the wheel contact points
* $r $: Operating radius of omni-wheel tires
* $\alpha_f$ = : Front wheel mounting angle relative to local $+x_r$ axis
* $\alpha_b $= : Rear wheel mounting angle relative to local $+x_r$ axis
* $h$: Height of Center of Mass (CoM) above ground plane
* $y_{cg}$ : Center of Mass position forward of the base centre (0 for now: centred)
* $V_{max}$ : Supply voltage saturation limit
* $R$ , $L_m$ : DC motor armature winding resistance and inductance
* $K_t $= $K_b$ : Electromechanical torque and back-EMF constants
* $C_{rr}$ : Floor-tire rolling resistance coefficient
* $\mu_s$ , $\mu_k$ : Static and kinetic floor-tire friction coefficients
* $\delta$ : LuGre presliding bristle deflection limit ($2\,\mu\text{m}$)
* $\zeta $: LuGre bristle damping ratio fraction against rim inertia
* $v_\sigma$ : Slip velocity threshold where bristle damping fades out
* $v_c $: Creep regularization velocity preventing division by zero at rest
* $\sigma_2$ : LuGre viscous friction coefficient, **set to 0**: the term stays in item 6 of section 5 but has no effect
* $\alpha_d $: Stribeck exponential decay rate from static to kinetic friction
* $\varepsilon $: Hyperbolic tangent smoothing speed for rolling resistance drag
* $\mu_{a,s}$, $\mu_{a,k}$, $\alpha_a$, $k_a$ : Roller (a-axis) friction coefficients at zero and at high axle slip, their decay rate, and the slope of the smoothed sign (change 9)
* $\epsilon_{tol}$, $n_{max}$ : Tolerance on the slip change, and the most solves per step, for the wheel solve (`IMPLICIT_TOL`, `IMPLICIT_MAX_ITER`)

---

### 2. Signal Tracker Definitions

At each sample time step $k$, the discrete simulation cycle processes the following variables:

#### System Inputs
* $\mathbf{V}_k = [V_{1,k}, V_{2,k}, V_{3,k}, V_{4,k}]^\top$: Armature terminal voltage commands (Volts), clipped to $\pm V_{max}$.

#### Tracked Memory Integration States
* $\mathbf{I}_k = [i_{1,k}, i_{2,k}, i_{3,k}, i_{4,k}]^\top$: Motor armature currents (Amperes).
* $\mathbf{\Omega}_k = [\omega_{1,k}, \omega_{2,k}, \omega_{3,k}, \omega_{4,k}]^\top$: Wheel angular velocities (rad/s).
* $\mathbf{z}_k = [z_{1,k}, z_{2,k}, z_{3,k}, z_{4,k}]^\top$: LuGre normalized bristle deflections (meters / normalized).
* $\mathbf{\dot{X}}_{r,k} = [\dot{x}_{r,k}, \dot{y}_{r,k}, \dot{\theta}_{k}]^\top$: Local chassis velocities (m/s Right, m/s Forward, rad/s Yaw), derived from the global velocities.
* $\mathbf{\ddot{X}}_{r,k-1} = [a_{x,r,k-1}, a_{y,r,k-1}, \alpha_{r,k-1}]^\top$: Historical chassis accelerations from step $k-1$ (m/s², m/s², rad/s²).
* $\mathbf{\dot{X}}_{g,k} = [\dot{x}_{g,k}, \dot{y}_{g,k}, \dot{\theta}_{g,k}]^\top$: Global room frame velocities (m/s, m/s, rad/s): the advanced velocity state, since Newton's law holds in this fixed frame.
* $\mathbf{X}_{g,k} = [x_k, y_k, \theta_k]^\top$: Global room frame navigation coordinates (meters, meters, radians).

---

### 3. Kinematic Mapping Matrix $\mathbf{H}$ (Body Frame: $+x_r$ Right, $+y_r$ Forward)

The chassis motion is measured about the Center of Mass, $y_{cg}$ forward of the base centre; each wheel's yaw lever about it is $R_b - y_{cg}\sin\beta_i$ ($R_b$ while the CoG is centred):

$$\mathbf{v}_{w,k} = \mathbf{H} \mathbf{\dot{X}}_{r,k}, \qquad \mathbf{H} = \begin{bmatrix} -\sin\alpha_f & \cos\alpha_f & R_b - y_{cg}\sin\alpha_f \\ -\sin\alpha_f & -\cos\alpha_f & R_b - y_{cg}\sin\alpha_f \\ \sin\alpha_b & -\cos\alpha_b & R_b + y_{cg}\sin\alpha_b \\ \sin\alpha_b & \cos\alpha_b & R_b + y_{cg}\sin\alpha_b \end{bmatrix}$$

$J_{zz}$ is the yaw inertia about the Center of Mass, so the mass matrix is $\mathbf{M} = \text{diag}(M_r, M_r, J_{zz})$.

---

### 4. Dynamic Normal Load Redistribution

With $+x_r$ Right (strafing) and $+y_r$ Forward (longitudinal):
$$y_f = R_b \sin\alpha_f = 0.03975\text{ m}, \qquad y_b = R_b \sin\alpha_b = 0.05622\text{ m}, \qquad L = y_f + y_b = 0.09597\text{ m}$$
$$x_f = R_b \cos\alpha_f = 0.06885\text{ m}, \qquad x_b = R_b \cos\alpha_b = 0.05622\text{ m}, \qquad \Sigma X^2 = x_f^2 + x_b^2 = 0.007901\text{ m}^2$$
The front and rear axle distances from the Center of Mass (equal to $y_f$, $y_b$ while it is centred):
$$y_{f,cg} = y_f - y_{cg}, \qquad y_{b,cg} = y_b + y_{cg}$$

#### Static Normal Force Distribution at Rest (values with the CoG centred)
$$F_{n,\text{front, static}} = \frac{M_r g y_{b,cg}}{2 L} = 7.752\text{ N}, \qquad F_{n,\text{rear, static}} = \frac{M_r g y_{f,cg}}{2 L} = 5.482\text{ N}$$

#### Dynamic Weight Shifts (Pitch due to $a_{y,\text{past}}$, Roll due to $a_{x,\text{past}}$)
$$\Delta F_{n,y} = \frac{M_r a_{y,\text{past}} h}{2 L}, \qquad \Delta F_{n,x,\text{front}} = \frac{x_f}{\Sigma X^2} \frac{M_r a_{x,\text{past}} h y_{b,cg}}{L}, \qquad \Delta F_{n,x,\text{rear}} = \frac{x_b}{\Sigma X^2} \frac{M_r a_{x,\text{past}} h y_{f,cg}}{L}$$

$$\mathbf{F}_{n,k} = \begin{bmatrix} F_{n,\text{front, static}} - \Delta F_{n,y} - \Delta F_{n,x,\text{front}} + w_{F,1} \\ F_{n,\text{front, static}} - \Delta F_{n,y} + \Delta F_{n,x,\text{front}} + w_{F,2} \\ F_{n,\text{rear, static}} + \Delta F_{n,y} + \Delta F_{n,x,\text{rear}} + w_{F,3} \\ F_{n,\text{rear, static}} + \Delta F_{n,y} - \Delta F_{n,x,\text{rear}} + w_{F,4} \end{bmatrix}$$

---

### 5. Motor, LuGre Bristle Friction & Discrete Integration

The motor current, wheel speed and bristle deflection of each wheel change faster than $\Delta T$ (the bristles' presliding and the motor's $L_m/R$), so they are discretised with **backward (implicit) Euler**, $\frac{dx}{dt} \approx \frac{x_{k+1} - x_k}{\Delta T}$ with the right-hand side at step $k+1$, which stays stable at any $\Delta T$. The three are solved together for each wheel (item 7). The chassis then follows with Forward Euler.

1. **Supply Limit and Motor Armature Current:**
   $$V_{i,k} = \text{clip}\left(V_{cmd,i,k}, -V_{max}, V_{max}\right), \qquad L_m \frac{i_{i,k+1} - i_{i,k}}{\Delta T} = V_{i,k} - R\, i_{i,k+1} - K_b\, \omega_{i,k+1}, \qquad T_{m,i,k+1} = K_t\, i_{i,k+1}$$

2. **Contact Slip Velocity along Rolling Axis** (chassis contact speed of step $k$, new wheel speed):
   $$v_{s,i,k+1} = (\mathbf{H} \mathbf{\dot{X}}_{r,k})_i - r \omega_{i,k+1}$$

3. **Smoothed Rolling Resistance Drag:**
   $$T_{\text{roll},i,k} = C_{rr} F_{n,i,k} r \tanh\left( \frac{\omega_{i,k}}{\varepsilon} \right)$$

4. **Stribeck Friction Curve:**
   $$\mu_{d,i} = \mu_k + (\mu_s - \mu_k) e^{-\alpha_d |v_{s,i,k+1}|}$$

5. **Backward Euler Bristle Update ($\Delta T = 1\text{ ms}$):**
   $$\delta \, \frac{z_{i,k+1} - z_{i,k}}{\Delta T} = v_{s,i,k+1} - \frac{\mu_s}{\mu_{d,i}} \sqrt{v_{s,i,k+1}^2 + v_c^2}\; z_{i,k+1}$$

6. **Net Tractive Floor Force:**
   $$\sigma_{1,i} = 2 \zeta \sqrt{ \frac{\mu_s F_{n,i,k}}{\delta} \frac{J_w}{r^2} } e^{-(v_{s,i,k+1} / v_\sigma)^2}$$
   $$F_{i,k+1} = -\left( \mu_s F_{n,i,k} z_{i,k+1} + \sigma_{1,i} \delta \frac{z_{i,k+1} - z_{i,k}}{\Delta T} + \sigma_2 v_{s,i,k+1} \right)$$
   With $\sigma_2 = 0$, the last term of $F_{i,k+1}$ is zero.
   In sliding the bristles settle within a step, $F_i = -\text{sign}(v_{s,i})\, \mu_{d,i} F_{n,i}$: the Coulomb–Stribeck law.

7. **Unified Wheel Speed Update (solved with items 1, 5 and 6):**
   $$J_w \frac{\omega_{i,k+1} - \omega_{i,k}}{\Delta T} = T_{m,i,k+1} - T_{\text{roll},i,k} - r F_{i,k+1}$$
   With $\mu_{d,i}$ and $\sigma_{1,i}$ fixed, $i_{i,k+1}$, $z_{i,k+1}$ and $F_{i,k+1}$ are linear in $\omega_{i,k+1}$, so this is one linear equation per wheel. $\mu_{d,i}$ and $\sigma_{1,i}$ depend on the new slip, so the solve is repeated with them updated at the new slip until the slip changes by less than $\epsilon_{tol}$ (at most $n_{max}$ solves). The load torque on each motor is $T_{load,i,k} = r F_{i,k+1} + T_{\text{roll},i,k}$.

8. **Roller (a-axis) Friction — explicit, from the chassis velocity of step $k$ (change 9):** each wheel's rollers let it slide along its axle direction $\mathbf{a}_i$ (the drive direction $\mathbf{d}_i$ turned $-90^\circ$), and the roller bushings resist that sliding. With $\mathbf{p}_i$ the wheel's contact point from the Center of Mass:
   $$v_{a,i,k} = \mathbf{a}_i \cdot \left( \begin{bmatrix} \dot{x}_{r,k} \\ \dot{y}_{r,k} \end{bmatrix} + \dot{\theta}_k \begin{bmatrix} -p_{y,i} \\ p_{x,i} \end{bmatrix} \right), \qquad \mu_{a,i} = \mu_{a,k} + (\mu_{a,s} - \mu_{a,k})\, e^{-\alpha_a |v_{a,i,k}|}$$
   $$F_{a,i,k} = -F_{n,i,k}\, \mu_{a,i}\, \frac{2}{\pi} \arctan\left(k_a\, v_{a,i,k}\right), \qquad \mathbf{W}_{a,k} = \sum_i \begin{bmatrix} F_{a,i,k}\, \mathbf{a}_i \\ \left(\mathbf{p}_i \times F_{a,i,k}\, \mathbf{a}_i\right)_z \end{bmatrix}$$
   The velocity of step $k$ is held over the step: a one-step lag, which is stable because this force is not stiff (it changes the chassis velocity at most about 160 /s, against $1/\Delta T$ = 1000 /s). The roller force lies along the wheel's axle, so it does not turn the wheel; it acts on the chassis only. **Off by default:** `ROLLER_FRICTION` in the plant configuration cell.

9. **Rigid Chassis Kinetics (global room frame):** the wheels are fixed to the chassis, so their forces act along body directions; Newton's law holds in the fixed room frame, so the global velocities are advanced there with the heading $\theta_k$:
   $$\mathbf{\ddot{X}}_{r,k} = \mathbf{M}^{-1} \left( \mathbf{H}^\top \mathbf{F}_{k+1} + \mathbf{W}_{a,k} \right), \qquad \mathbf{\dot{X}}_{g,k+1} = \mathbf{\dot{X}}_{g,k} + \Delta T\, \mathbf{R}(\theta_k)\, \mathbf{\ddot{X}}_{r,k}, \qquad \mathbf{X}_{g,k+1} = \mathbf{X}_{g,k} + \Delta T\, \mathbf{\dot{X}}_{g,k+1}$$
   $$\mathbf{R}(\theta) = \begin{bmatrix} \cos\theta & -\sin\theta & 0 \\ \sin\theta & \cos\theta & 0 \\ 0 & 0 & 1 \end{bmatrix}, \qquad \mathbf{\dot{X}}_{r,k+1} = \mathbf{R}(\theta_{k+1})^\top \mathbf{\dot{X}}_{g,k+1}$$
   The local velocities are not advanced themselves: the local axes turn with the robot, so advancing them directly leaves out the $-\boldsymbol\omega \times \mathbf{\dot{X}}_r$ term, and a robot that moves while it turns follows a wrong, curved path.

</details>

---
<details>
<summary><b> Open this for simulation flowchart </b></summary>


```text
==================================================
        SIMULATION STATE DATA (STEP K)
==================================================
 [Inputs] -> Voltages Vector V_1..4
 [Memory] -> Current i_k, Wheel Speeds omega_k
 [Memory] -> LuGre Bristle Deflections z_k
 [Memory] -> Global Chassis Speeds & Past Accel (a_x, a_y)
 [Memory] -> Global Coordinates [x, y, theta]
==================================================
                        |
                        v
  +--------------------------------------------+
  | Step 5.1: Motor Armature Current Update    |
  | V = clip(V_cmd, +-V_max)                   |
  | Backward Euler: i_{k+1} in terms of        |
  | omega_{k+1}, T_m = Kt * i_{k+1}            |
  +--------------------------------------------+
                        |
                        v
  +--------------------------------------------+
  | Step 5.2: Dynamic Normal Force Distribution|
  | (x Right, y Forward Frame)                 |
  | Static load + Pitch (ay) + Roll (ax) + w_F |
  +--------------------------------------------+
                        |
                        v
  +--------------------------------------------+
  | Step 5.3: LuGre Bristle Friction Update    |
  | v_s = H*v_local_k - r*omega_{k+1}          |
  | Backward Euler: z_{k+1}, F_floor in terms  |
  | of omega_{k+1}                             |
  +--------------------------------------------+
                        |
                        v
  +--------------------------------------------+
  | Step 5.6: Wheel Speed Update               |
  | Jw*(omega_{k+1}-omega_k)/dt =              |
  |      T_m - T_roll - r*F_floor              |
  | Solved with 5.1 and 5.3; repeated until    |
  | the slip settles                           |
  +--------------------------------------------+
                        |
                        v
  +--------------------------------------------+
  | Step 5.8: Roller (a-axis) Friction         |
  | explicit, chassis velocity of step k       |
  | v_a = a_i . (v + theta_dot x p_i)          |
  | F_a = -F_n*mu_a*(2/pi)*atan(k_a*v_a)       |
  | -> wrench W_roller on the chassis only     |
  +--------------------------------------------+
                        |
                        v
  +--------------------------------------------+
  | Step 5.4 & 5.5: Rigid Chassis Kinetics     |
  | a_local = M_inv*(H^T*F_floor + W_roller)   |
  | v_global_{k+1} = v_global_k                |
  |                  + dt * R(theta_k)*a_local |
  | pos_{k+1} = pos_k + dt * v_global_{k+1}    |
  | v_local_{k+1} = R(theta_{k+1})^T v_global  |
  +--------------------------------------------+
                        |
                        v
==================================================
        ADVANCE STATE TO STEP K+1
==================================================
```
</details>

### 📂 Omni Simulation Architecture

The code is split into four parts, each in its own cell below, executed at each time step $\Delta T$:

```text
Controller  ->  Simulation  ->  Timestep  ->  Model
what voltages   run the loop,   advance by    the physics: no time step,
to apply        draw the noise  one step      no noise generator
```

**Model:** the plant physics.

*   **`wheel_angles`, `wheel_frame`, `wheel_geometry`**: The wheel directions, lever arms ($x_f, x_b, y_f, y_b$, and the front and rear distances from the CoG) and the $4 \times 3$ kinematic transformation matrix $\mathbf{H}$, with each wheel's yaw lever about the CoG.
*   **`local_to_global`, `global_to_local`**: The rotations between the local chassis frame and the global room frame.
*   **`wheel_center_velocity`, `slip_velocity`**: The velocity of each wheel centre, then the contact point velocity and its slip along the drive and axle directions.
*   **`wheel_normal_forces`**: Evaluates vertical normal loads $\mathbf{F}_{n,k}$ with pitch/roll load shifts based on past accelerations ($a_{x,\text{past}}, a_{y,\text{past}}$) and the vibration disturbance it is given.
*   **`drive_friction_coefficients`, `mu_d`, `lugre_coefficients`**: The Stribeck friction coefficient, static limit and bristle damping for given slip speeds.
*   **`rolling_resistance`**: The smoothed rolling drag.
*   **`mu_a`, `roller_friction`**: Roller (a-axis) friction: each wheel's roller force along the axle, and their wrench on the chassis (change 9).
*   **`chassis_acceleration`, `wheel_load_torque`**: Newton-Euler rigid chassis acceleration, using the inverse rigid-body mass matrix $\mathbf{M}^{-1}$, and the load torque on each motor.

**Timestep:** the discrete-time steps.

*   **`motor_current_line`**: The new motor currents $\mathbf{I}_{k+1}$ (backward Euler), as linear forms in the new wheel speeds.
*   **`bristle_lines`**: The new LuGre bristle deflections $\mathbf{z}_{k+1}$ and floor forces (backward Euler), as linear forms in the new wheel speeds.
*   **`solve_wheels`**: Solves each wheel's torque balance for its new speed, repeated until the slip settles.
*   **`advance_chassis`**: Rotates the accelerations into the global room frame, advances the global velocities, then the coordinates.
*   **`initial_state`**: The memory states (armature currents, wheel speeds, LuGre bristle deflections, local/global velocities, position coordinates, and the applied voltages) carried from step $k$ to $k+1$.
*   **`step`**: Executes the master pipeline for one discrete time step $\Delta T$: clips the voltages to the supply, then runs the functions above in order.

**Simulation:** the loop around the plant.

*   **`draw_disturbances`**: The noise for one step.
*   **`measured_outputs`**: What the controller is allowed to see: $[X, Y, \theta, \omega_1, \dots, \omega_4]$.
*   **`initialize_history`, `record_history`**: The logged signals for the plots.
*   **`run_simulation`**: One run: measure, ask the controller, draw the noise, step, record.
*   **`run_ideal_and_noisy`**: The same run with the noise off, then on.

**Controller:** the voltage commands.

*   **`constant_voltage`, `multi_stage_voltage`**: The open-loop test commands. The MPC goes here.

In [ ]:
"""
Model
=====

The plant physics: the robot's geometry and the forces on it.

Nothing here knows about the time step, the noise generator or the controller. Disturbance
values come in as plain inputs (w_F, w_mu); the simulation is what draws them.

Flow:

    wheel layout
        ↓
    wheel normal loads
        ↓
    friction: drive (LuGre), rolling resistance, rollers
        ↓
    chassis acceleration

Conventions:
    Frames: x right, y forward. "local" moves with the robot, "global" is fixed to the room.
    theta: robot heading, counter-clockwise from the global x axis [rad]
    omega: wheel speeds [rad/s]
    H: the 4x3 matrix from the spec, which maps [x_dot, y_dot, theta_dot] to the speed of each wheel
    Arrays with one value per wheel are in the order 1 FR, 2 FL, 3 RL, 4 RR.

Velocities, named as in archive/williams/williams_model.py:
    v_local, v_global: the chassis at the CoG, in each frame [x_dot, y_dot, theta_dot]
    v_center: each wheel's centre = v_local + theta_dot x r_i. No wheel spin
    v_center_d: v_center along d_hat = H * v_local
    r * omega: the rim speed from the wheel's spin, along d_hat
    v_contact: the wheel's contact point on the floor = v_center - r omega d_hat
    v_slip_d: v_contact along d_hat = v_center_d - r omega
    v_slip_a: v_contact along a_hat = v_center . a_hat
Here a positive omega drives the robot along +d_hat, so the rim term is subtracted. Williams'
theta_dot has the opposite sign, which is why his V_contact adds it.

Spec cell references are given as "spec 5.1" = section 5, item 1. Where a name differs from the
spec, its docstring gives the spec symbol in brackets.
The [CHANGE n] tags point to the entries in dana-models/open_loop_model_changes.md.
"""

import numpy as np

# ---------------------------------------------------------------------------
# Wheel geometry
# ---------------------------------------------------------------------------

def wheel_angles(front_angle, rear_angle):
    """Position of each wheel around the base circle, counter-clockwise from +x.

    front_angle: angle of the front wheels above the x axis (alpha_f) [rad]
    rear_angle: angle of the rear wheels below the x axis (alpha_b) [rad]

    Returns the four wheel angles [rad].
    """
    return np.array([front_angle, np.pi - front_angle, np.pi + rear_angle, 2.0 * np.pi - rear_angle])


def wheel_frame(angles):
    """Unit vectors of each wheel, local frame.

    angles: wheel positions from wheel_angles [rad]

    Returns (d_hat, a_hat), each with one [x, y] row per wheel:
        d_hat: the way the wheel pushes, the tangent of the base circle
        a_hat: the way the rollers slide, radial for these tangential wheels
    """
    d_hat = np.column_stack((-np.sin(angles), np.cos(angles)))
    a_hat = np.column_stack((np.cos(angles), np.sin(angles)))
    return d_hat, a_hat


def wheel_geometry(front_angle, rear_angle):
    """Everything fixed by the wheel layout and the CoG position. [CHANGE 1, 6, 9]

    front_angle: angle of the front wheels above the x axis (alpha_f) [rad]
    rear_angle: angle of the rear wheels below the x axis (alpha_b) [rad]

    Returns a dictionary:
        H: rows are [drive x, drive y, yaw lever], so wheel speeds = H * [x_dot, y_dot, theta_dot] (spec 3)
        d_hat, a_hat: drive and axle direction of each wheel
        positions: [x, y] of each wheel from the CoG (r_i) [m]
        x_f, x_b: sideways distance of each axle's wheels from the centreline [m]
        y_f, y_b: distance of each axle from the CoG, for the load split (spec 4) [m]
        wheelbase: distance between the two axles [m]
        sum_x2: x_f^2 + x_b^2, for the roll load shift [m^2]
    """
    angles = wheel_angles(front_angle, rear_angle)
    d_hat, a_hat = wheel_frame(angles)

    # A forward CoG shortens the yaw lever of the front wheels and lengthens the rear ones
    yaw_lever = WHEEL_ROBOT_DIST - COG_FORWARD * np.sin(angles)
    positions = WHEEL_ROBOT_DIST * a_hat - np.array([0.0, COG_FORWARD])

    x_f = WHEEL_ROBOT_DIST * np.cos(front_angle)
    x_b = WHEEL_ROBOT_DIST * np.cos(rear_angle)
    y_f = WHEEL_ROBOT_DIST * np.sin(front_angle)    # from the base centre
    y_b = WHEEL_ROBOT_DIST * np.sin(rear_angle)      # from the base centre

    return {
        "H": np.column_stack((d_hat, yaw_lever)),
        "d_hat": d_hat,
        "a_hat": a_hat,
        "positions": positions,
        "x_f": x_f,
        "x_b": x_b,
        "y_f": y_f - COG_FORWARD,
        "y_b": y_b + COG_FORWARD,
        "wheelbase": y_f + y_b,
        "sum_x2": x_f**2 + x_b**2,
    }


# ---------------------------------------------------------------------------
# Frame rotations
# ---------------------------------------------------------------------------

def local_to_global(vector, theta):
    """Rotate a vector from the local frame into the global frame. Yaw is unchanged.

    vector: [x, y, yaw]
    theta: robot heading [rad]
    """
    return np.array([np.cos(theta) * vector[0] - np.sin(theta) * vector[1],
                     np.sin(theta) * vector[0] + np.cos(theta) * vector[1],
                     vector[2]])


def global_to_local(vector, theta):
    """Rotate a vector from the global frame into the local frame. Yaw is unchanged.

    vector: [x, y, yaw]
    theta: robot heading [rad]
    """
    return np.array([np.cos(theta) * vector[0] + np.sin(theta) * vector[1],
                     -np.sin(theta) * vector[0] + np.cos(theta) * vector[1],
                     vector[2]])


# ---------------------------------------------------------------------------
# Wheel kinematics
# ---------------------------------------------------------------------------

def wheel_center_velocity(v_local, positions):
    """Velocity of each wheel centre, local frame. It does not include the wheel's spin.

    v_center = v_local + theta_dot x r_i

    v_local: chassis velocity at the CoG, local frame [x_dot, y_dot, theta_dot]
    positions: [x, y] of each wheel from the CoG (r_i) [m]

    Returns one [x, y] row per wheel [m/s].
    """
    return np.column_stack((v_local[0] - v_local[2] * positions[:, 1],
                            v_local[1] + v_local[2] * positions[:, 0]))


def slip_velocity(v_center, omega, geometry):
    """Slip of each wheel along its drive and axle directions.

    v_contact = v_center - r omega d_hat
    v_slip_d = v_contact . d_hat
    v_slip_a = v_contact . a_hat

    v_center: from wheel_center_velocity [m/s]
    omega: wheel speeds [rad/s]
    geometry: from wheel_geometry

    Returns (v_slip_d, v_slip_a) [m/s].
    """
    d_hat, a_hat = geometry["d_hat"], geometry["a_hat"]

    v_contact = v_center - WHEEL_RADIUS * omega[:, None] * d_hat

    return np.sum(v_contact * d_hat, axis=1), np.sum(v_contact * a_hat, axis=1)


# ---------------------------------------------------------------------------
# Wheel normal loads
# ---------------------------------------------------------------------------

def wheel_normal_forces(a_local_past, w_F, geometry):
    """Vertical load on each wheel (F_n, spec 4). [CHANGE 1, 6]

    a_local_past: chassis acceleration of the previous step, local frame [x, y, yaw]. Using the
                  previous step avoids an algebraic loop: the loads set the forces, which set
                  the acceleration.
    w_F: load disturbance on each wheel [N]
    geometry: from wheel_geometry

    Returns the load on each wheel [N].
    """
    y_f, y_b = geometry["y_f"], geometry["y_b"]
    wheelbase = geometry["wheelbase"]

    # Static split: each axle carries more when the CoG is closer to it
    front_static = (ROBOT_MASS * GRAVITY * y_b) / (2.0 * wheelbase)
    rear_static = (ROBOT_MASS * GRAVITY * y_f) / (2.0 * wheelbase)

    # Pitch: accelerating forward moves load from the front wheels to the rear
    pitch_shift = (ROBOT_MASS * a_local_past[1] * COG_HEIGHT) / (2.0 * wheelbase)

    # Roll: accelerating right moves load from the right wheels to the left
    roll_moment = (ROBOT_MASS * a_local_past[0] * COG_HEIGHT) / wheelbase
    roll_shift_front = (geometry["x_f"] / geometry["sum_x2"]) * roll_moment * y_b
    roll_shift_rear = (geometry["x_b"] / geometry["sum_x2"]) * roll_moment * y_f

    F_n = np.array([
        front_static - pitch_shift - roll_shift_front + w_F[0],   # 1 FR
        front_static - pitch_shift + roll_shift_front + w_F[1],   # 2 FL
        rear_static + pitch_shift + roll_shift_rear + w_F[2],     # 3 RL
        rear_static + pitch_shift - roll_shift_rear + w_F[3],     # 4 RR
    ])

    # A wheel cannot pull on the floor
    return np.maximum(0.0, F_n)


# ---------------------------------------------------------------------------
# Drive friction
# ---------------------------------------------------------------------------

def drive_friction_coefficients(w_mu):
    """Floor friction coefficients of each wheel for this step.

    w_mu: friction disturbance on each wheel

    Returns (mu_s, mu_k), neither below zero.
    """
    mu_s = np.maximum(0.0, MU_D_STAT + w_mu)
    mu_k = np.maximum(0.0, MU_D_DYN + w_mu)
    return mu_s, mu_k


def mu_d(v_slip_d, mu_s, mu_k):
    """Drive-axis Stribeck curve: mu_s at zero slip, decaying to mu_k. [CHANGE 7]

    v_slip_d: slip of each wheel along its drive direction [m/s]
    mu_s, mu_k: from drive_friction_coefficients
    """
    return mu_k + (mu_s - mu_k) * np.exp(-MU_D_DECAY * np.abs(v_slip_d))


def lugre_coefficients(v_slip_d, F_n, mu_s, mu_k):
    """LuGre terms at this slip (spec 5.4-5.6). [CHANGE 7]

    v_slip_d: slip of each wheel along its drive direction [m/s]
    F_n: vertical load on each wheel [N]
    mu_s, mu_k: from drive_friction_coefficients

    Returns (F_static, bristle_decay, bristle_damping):
        F_static: largest force the bristles can hold, mu_s * F_n [N]
        bristle_decay: how fast slip relaxes the bristles (a) [m/s]
        bristle_damping: damping of the bristles, fading out with slip (sigma_1)
    """
    F_static = mu_s * F_n
    bristle_decay = (mu_s / np.maximum(mu_d(v_slip_d, mu_s, mu_k), 1e-9)) * np.sqrt(v_slip_d**2 + LUGRE_CREEP**2)
    bristle_damping = (2.0 * LUGRE_DAMPING_RATIO * np.sqrt(F_static / LUGRE_DEFLECTION * RIM_MASS)
                       * np.exp(-(v_slip_d / LUGRE_DAMPING_SLIP)**2))
    return F_static, bristle_decay, bristle_damping


# ---------------------------------------------------------------------------
# Rolling resistance
# ---------------------------------------------------------------------------

def rolling_resistance(omega, F_n):
    """Rolling drag torque on each wheel, smoothed through zero speed [N m] (spec 5.3). [CHANGE 7]

    omega: wheel speeds [rad/s]
    F_n: vertical load on each wheel [N]
    """
    return WHEEL_ROLLING_RESISTANCE_COEFF * F_n * WHEEL_RADIUS * np.tanh(omega / WHEEL_LOSS_SMOOTHING_SPEED)


# ---------------------------------------------------------------------------
# Roller friction
# ---------------------------------------------------------------------------

def mu_a(v_slip_a):
    """Axle-axis (roller) friction coefficient: MU_A_STAT at zero slip, tending to MU_A_DYN. [CHANGE 9]

    v_slip_a: slip of each wheel along its axle [m/s]
    """
    return MU_A_DYN + (MU_A_STAT - MU_A_DYN) * np.exp(-MU_A_DECAY * np.abs(v_slip_a))


def roller_friction(v_slip_a, F_n, geometry):
    """Roller friction on the chassis (spec 5.8). [CHANGE 9]

    It lies along each wheel's axle, so it cannot turn the wheel.

    v_slip_a: slip of each wheel along a_hat [m/s]
    F_n: vertical load on each wheel [N]
    geometry: from wheel_geometry

    Returns (wrench_roller, F_a):
        wrench_roller: [force x, force y, moment] on the chassis about the CoG [N, N, N m]
        F_a: force of each wheel along a_hat [N]
    """
    if not ROLLER_FRICTION:
        return np.zeros(3), np.zeros(4)

    a_hat, positions = geometry["a_hat"], geometry["positions"]

    # (2/pi) atan(k v) is a smoothed sign(v)
    F_a = -F_n * mu_a(v_slip_a) * (2.0 / np.pi) * np.arctan(K_SMOOTH_A * v_slip_a)

    # Add up the wheels' forces and their moments about the CoG
    F_vectors = F_a[:, None] * a_hat
    moment = np.sum(positions[:, 0] * F_vectors[:, 1] - positions[:, 1] * F_vectors[:, 0])

    return np.array([F_vectors[:, 0].sum(), F_vectors[:, 1].sum(), moment]), F_a


# ---------------------------------------------------------------------------
# Chassis
# ---------------------------------------------------------------------------

def chassis_acceleration(F_floor, wrench_roller, H):
    """Chassis acceleration, local frame [x, y, yaw] (spec 5.9).

    F_floor: force of the floor on each wheel, along its drive direction [N]
    wrench_roller: from roller_friction
    H: from wheel_geometry
    """
    return MASS_MATRIX_INV @ (H.T @ F_floor + wrench_roller)


def wheel_load_torque(F_floor, T_roll):
    """Load torque on each motor [N m].

    F_floor: force of the floor on each wheel, along its drive direction [N]
    T_roll: from rolling_resistance [N m]
    """
    return F_floor * WHEEL_RADIUS + T_roll

In [ ]:
"""
Timestep
========

Advances the model by one DT: step(state, V_cmd, geometry, w_F, w_mu).

Flow:

    voltage command
        ↓
    supply clip
        ↓
    wheel normal loads (from the previous step's acceleration)
        ↓
    roller friction (from the chassis velocity of this step's start)
        ↓
    wheel solve: motor current, wheel speed and LuGre bristles together (backward Euler)
        ↓
    chassis acceleration
        ↓
    global velocity, then pose (forward Euler)

The bristles and the wheel form a stiff loop (w*dt ~ 3.8), so the wheel solve is implicit. [CHANGE 5]
The new current, bristle state and floor force are each written as a line in the new wheel speed:

    I_next = I_stalled + dI_domega * omega_next
    z_next = z_stalled + dz_domega * omega_next
    F_next = F_stalled + dF_domega * omega_next

where each "stalled" value is what it would be if the wheel were stopped at the end of the step,
and each slope is how much a rad/s of wheel speed changes it.

and the wheel torque balance is then one linear equation for omega_next.
"""

import numpy as np

# ---------------------------------------------------------------------------
# Wheel solve: backward Euler
# ---------------------------------------------------------------------------

def motor_current_line(I, V):
    """The new motor current, written as a straight line in the new wheel speed (spec 5.1).

    The motor circuit at the end of the step (backward Euler):

        L (I_next - I) / DT = V - R I_next - Kb omega_next

    omega_next is not known yet, so I_next can only be written in terms of it. Solving for I_next:

        I_next = (L / DT * I + V - Kb omega_next) / (L / DT + R)
               = I_stalled + dI_domega * omega_next

    I: motor currents at the start of the step [A]
    V: applied motor voltages [V]

    Returns (I_stalled, dI_domega):
        I_stalled: the new current if the wheel were stopped, so with no back-EMF [A]
        dI_domega: how much each rad/s of wheel speed changes the new current, through the back-EMF [A/(rad/s)]
    """
    L_over_DT = MOTOR_INDUCTANCE / DT
    denominator = L_over_DT + MOTOR_RESISTANCE

    I_stalled = (L_over_DT * I + V) / denominator
    dI_domega = -MOTOR_KB / denominator
    return I_stalled, dI_domega


def bristle_lines(v_slip_d, z, v_center_d, F_n, mu_s, mu_k):
    """The new bristle state and floor force, each written as a straight line in the new wheel speed (spec 5.4-5.6). [CHANGE 7]

    The bristles and the floor force at the end of the step (backward Euler):

        delta (z_next - z) / DT = v_slip_d_next - bristle_decay z_next
        F_next = -(F_static z_next + bristle_damping delta (z_next - z) / DT + sigma_2 v_slip_d_next)
        v_slip_d_next = v_center_d - r omega_next

    omega_next is not known yet, so z_next and F_next can only be written in terms of it:

        z_next = z_stalled + dz_domega * omega_next
        F_next = F_stalled + dF_domega * omega_next

    v_slip_d: the drive slip at which the Stribeck coefficient and the bristle damping are evaluated [m/s]
    z: bristle deflections at the start of the step, as a fraction of LUGRE_DEFLECTION
    v_center_d: wheel centre velocity along d_hat (H * v_local) [m/s]
    F_n: vertical load on each wheel [N]
    mu_s, mu_k: from drive_friction_coefficients

    Returns (z_stalled, dz_domega, F_stalled, dF_domega):
        z_stalled: the new bristle deflection if the wheel were stopped at the end of the step
        dz_domega: how much each rad/s of wheel speed changes the new bristle deflection [1/(rad/s)]
        F_stalled: the floor force if the wheel were stopped at the end of the step [N]
        dF_domega: how much each rad/s of wheel speed changes the floor force [N/(rad/s)]
    """
    F_static, bristle_decay, bristle_damping = lugre_coefficients(v_slip_d, F_n, mu_s, mu_k)

    denominator = 1.0 + bristle_decay * DT / LUGRE_DEFLECTION
    z_stalled = (z + DT / LUGRE_DEFLECTION * v_center_d) / denominator
    dz_domega = -(DT / LUGRE_DEFLECTION) * WHEEL_RADIUS / denominator

    F_stalled = -(F_static * z_stalled + bristle_damping * LUGRE_DEFLECTION * (z_stalled - z) / DT + LUGRE_VISCOUS * v_center_d)
    dF_domega = -(F_static * dz_domega + bristle_damping * LUGRE_DEFLECTION * dz_domega / DT - LUGRE_VISCOUS * WHEEL_RADIUS)
    return z_stalled, dz_domega, F_stalled, dF_domega


def solve_wheels(state, V, v_center, geometry, F_n, mu_s, mu_k):
    """Solve current, wheel speed and bristles together for each wheel (spec 5.7). [CHANGE 5, 7]

    The Stribeck coefficient and the damping depend on the new slip, so the solve is repeated
    with them updated until the slip settles.

    state: robot state at the start of the step
    V: applied motor voltages [V]
    v_center: from wheel_center_velocity [m/s]
    geometry: from wheel_geometry
    F_n: vertical load on each wheel [N]
    mu_s, mu_k: from drive_friction_coefficients

    Returns (I_next, omega_next, z_next, F_floor, T_roll, solves):
        F_floor: force of the floor on each wheel, along its drive direction [N]
        T_roll: rolling drag on each wheel [N m]
        solves: how many times the solve ran
    """
    omega, z = state["omega"], state["z"]
    v_center_d = geometry["H"] @ state["v_local"]   # v_center along d_hat

    I_stalled, dI_domega = motor_current_line(state["I"], V)
    T_roll = rolling_resistance(omega, F_n)

    v_slip_d, _ = slip_velocity(v_center, omega, geometry)
    for solves in range(1, IMPLICIT_MAX_ITER + 1):
        z_stalled, dz_domega, F_stalled, dF_domega = bristle_lines(v_slip_d, z, v_center_d, F_n, mu_s, mu_k)

        # Torque balance: J (omega_next - omega) / DT = Kt I_next - T_roll - r F_next
        omega_next = ((WHEEL_INERTIA / DT * omega + MOTOR_KT * I_stalled - T_roll - WHEEL_RADIUS * F_stalled)
                      / (WHEEL_INERTIA / DT - MOTOR_KT * dI_domega + WHEEL_RADIUS * dF_domega))

        # Stop once the new slip has stopped changing
        v_slip_d_next, _ = slip_velocity(v_center, omega_next, geometry)
        settled = np.max(np.abs(v_slip_d_next - v_slip_d)) < IMPLICIT_TOL
        v_slip_d = v_slip_d_next
        if settled:
            break

    I_next = I_stalled + dI_domega * omega_next
    z_next = z_stalled + dz_domega * omega_next
    F_floor = F_stalled + dF_domega * omega_next
    return I_next, omega_next, z_next, F_floor, T_roll, solves


# ---------------------------------------------------------------------------
# Chassis update: forward Euler
# ---------------------------------------------------------------------------

def advance_chassis(v_global, pos_global, a_local):
    """Advance the velocity, then the position, in the global frame (spec 5.9). [CHANGE 2]

    Newton's law holds in the global frame, so the global velocity is the state. Advancing the
    local velocity instead (Dana's original) leaves out -omega x v, and a robot that moves
    while it turns follows a wrong curved path.

    v_global: global velocity [x_dot, y_dot, theta_dot]
    pos_global: pose [X, Y, theta]
    a_local: chassis acceleration, local frame [x, y, yaw]

    Returns (v_global_next, pos_global_next).
    """
    v_global_next = v_global + DT * local_to_global(a_local, pos_global[2])
    pos_global_next = pos_global + DT * v_global_next
    return v_global_next, pos_global_next


# ---------------------------------------------------------------------------
# One step
# ---------------------------------------------------------------------------

def initial_state():
    """Robot at rest at the origin."""
    return {
        "I": np.zeros(4),             # motor currents [A]
        "omega": np.zeros(4),         # wheel speeds [rad/s]
        "z": np.zeros(4),             # bristle deflections, as a fraction of LUGRE_DEFLECTION
        "v_global": np.zeros(3),      # global velocity [x_dot, y_dot, theta_dot], the advanced state
        "v_local": np.zeros(3),       # local velocity [x_dot, y_dot, theta_dot], derived from v_global
        "pos_global": np.zeros(3),    # pose [X, Y, theta] [m, m, rad]
        "a_local_past": np.zeros(3),  # chassis acceleration of the previous step, local frame [x, y, yaw]
        "V_actual": np.zeros(4),      # motor voltages after the supply clip [V]
    }


def step(state, V_cmd, geometry, w_F, w_mu):
    """Advance the robot by one DT, with the voltages and the disturbances held over the step.

    state: robot state at the start of the step, as in initial_state
    V_cmd: motor voltage commands [V]
    geometry: from wheel_geometry
    w_F: load disturbance on each wheel [N]
    w_mu: friction disturbance on each wheel

    Returns (next_state, T_load, slipping, solves):
        next_state: robot state at the end of the step
        T_load: load torque on each motor [N m]
        slipping: wheels whose slip this step is above DRIVE_BREAKAWAY_BAND (a report only)
        solves: how many times the wheel solve ran
    """
    V = np.clip(np.asarray(V_cmd, dtype=float), -V_MAX, V_MAX)   # [CHANGE 8]

    mu_s, mu_k = drive_friction_coefficients(w_mu)
    F_n = wheel_normal_forces(state["a_local_past"], w_F, geometry)

    v_center = wheel_center_velocity(state["v_local"], geometry["positions"])
    _, v_slip_a = slip_velocity(v_center, state["omega"], geometry)

    # Explicit, one step late. That is stable because this force is not stiff: at most about 160 /s
    # against 1/DT = 1000 /s.
    wrench_roller, _ = roller_friction(v_slip_a, F_n, geometry)

    I_next, omega_next, z_next, F_floor, T_roll, solves = solve_wheels(state, V, v_center, geometry, F_n, mu_s, mu_k)

    # The chassis responds to the new floor forces
    a_local = chassis_acceleration(F_floor, wrench_roller, geometry["H"])
    v_global_next, pos_global_next = advance_chassis(state["v_global"], state["pos_global"], a_local)

    next_state = {
        "I": I_next,
        "omega": omega_next,
        "z": z_next,
        "v_global": v_global_next,
        "v_local": global_to_local(v_global_next, pos_global_next[2]),
        "pos_global": pos_global_next,
        "a_local_past": a_local,
        "V_actual": V,
    }
    T_load = wheel_load_torque(F_floor, T_roll)
    v_slip_d, _ = slip_velocity(v_center, omega_next, geometry)
    slipping = np.abs(v_slip_d) > DRIVE_BREAKAWAY_BAND

    return next_state, T_load, slipping, solves

In [ ]:
"""
Simulation
==========

Runs the loop around the plant and applies the disturbances.

Flow:

    measured outputs [X, Y, theta, omega_1..4]
        ↓
    controller
        ↓
    voltage command
        ↓
    disturbance draws
        ↓
    plant step
        ↓
    history
"""

import numpy as np

# ---------------------------------------------------------------------------
# Disturbances
# ---------------------------------------------------------------------------

def draw_disturbances(rng, sigma_load, sigma_mu):
    """Noise for one step, one value per wheel.

    rng: random number generator
    sigma_load: standard deviation of the wheel load noise (sigma_V in the spec) [N], 0.0 for none
    sigma_mu: standard deviation of the friction coefficient noise, 0.0 for none

    Returns (w_F, w_mu).
    """
    w_F = rng.normal(0.0, sigma_load, 4) if sigma_load > 0 else np.zeros(4)
    w_mu = rng.normal(0.0, sigma_mu, 4) if sigma_mu > 0 else np.zeros(4)
    return w_F, w_mu


# ---------------------------------------------------------------------------
# Measurements
# ---------------------------------------------------------------------------

def measured_outputs(state):
    """All the controller gets to see: [X, Y, theta, omega_1, omega_2, omega_3, omega_4].

    state: robot state, as in initial_state
    """
    return np.concatenate((state["pos_global"], state["omega"]))


# ---------------------------------------------------------------------------
# History
# ---------------------------------------------------------------------------

def initialize_history(steps):
    """Empty arrays for the logged signals.

    steps: number of DT steps in the run
    """
    return {
        "time": np.arange(steps) * DT,          # [s]
        "V": np.zeros((steps, 4)),              # commanded motor voltages [V]
        "omega": np.zeros((steps, 4)),          # wheel speeds [rad/s]
        "v_local": np.zeros((steps, 3)),        # local velocity [x_dot, y_dot, theta_dot]
        "pos_global": np.zeros((steps, 3)),     # pose [X, Y, theta]
    }


def record_history(k, history, state, V_cmd):
    """Log one step.

    k: step number
    history: from initialize_history
    state: robot state at step k
    V_cmd: motor voltage commands at step k [V]
    """
    history["V"][k] = V_cmd
    history["omega"][k] = state["omega"]
    history["v_local"][k] = state["v_local"]
    history["pos_global"][k] = state["pos_global"]


# ---------------------------------------------------------------------------
# Main simulation loop
# ---------------------------------------------------------------------------

def run_simulation(controller, steps, geometry, sigma_load, sigma_mu):
    """One run. Every run starts at rest with the same random seed, so runs are repeatable.

    controller: function that returns the voltage command for (k, measurement)
    steps: number of DT steps
    geometry: from wheel_geometry
    sigma_load: standard deviation of the wheel load noise [N], 0.0 for none
    sigma_mu: standard deviation of the friction coefficient noise, 0.0 for none

    Returns the history of the run.
    """
    rng = np.random.default_rng(seed=RANDOM_SEED)
    state = initial_state()
    history = initialize_history(steps)

    for k in range(steps):
        V_cmd = controller(k, measured_outputs(state))
        record_history(k, history, state, V_cmd)

        w_F, w_mu = draw_disturbances(rng, sigma_load, sigma_mu)
        state, T_load, slipping, solves = step(state, V_cmd, geometry, w_F, w_mu)

    return history


def run_ideal_and_noisy(controller, steps, geometry):
    """The same run with the noise off, then on.

    controller: function that returns the voltage command for (k, measurement)
    steps: number of DT steps
    geometry: from wheel_geometry

    Returns (ideal, noisy), the two histories.
    """
    ideal = run_simulation(controller, steps, geometry, 0.0, 0.0)
    noisy = run_simulation(controller, steps, geometry, SIGMA_LOAD, SIGMA_MU)
    return ideal, noisy

In [ ]:
"""
Controller
==========

Decides the motor voltages. A controller is a function of (k, measurement) that returns the
four voltage commands [V]:

    k: step number
    measurement: [X, Y, theta, omega_1, omega_2, omega_3, omega_4], from measured_outputs

These are the open-loop test commands, so they ignore the measurement. The MPC goes here.
"""

import numpy as np

# Wheel voltage signs for each motion
FORWARD = np.array([1.0, -1.0, -1.0, 1.0])
STRAFE_RIGHT = np.array([-1.0, -1.0, 1.0, 1.0])
ROTATE_CCW = np.array([1.0, 1.0, 1.0, 1.0])


def constant_voltage(voltage, pattern):
    """A controller that holds one voltage for the whole run.

    voltage: voltage magnitude [V]
    pattern: wheel voltage signs, such as FORWARD
    """
    def controller(k, measurement):
        return voltage * pattern

    return controller


def multi_stage_voltage(voltage, pattern, steps_per_stage):
    """A controller that drives at +voltage, then reverses at -voltage, then coasts at 0 V.

    voltage: voltage magnitude [V]
    pattern: wheel voltage signs, such as FORWARD
    steps_per_stage: length of each of the three stages, in DT steps
    """
    def controller(k, measurement):
        if k < steps_per_stage:
            return voltage * pattern
        if k < 2 * steps_per_stage:
            return -voltage * pattern
        return np.zeros(4)

    return controller

```text
==========================================================================
                     SPATIAL REFERENCE FRAME LOGIC
==========================================================================

   GLOBAL FRAME (Fixed Room)               LOCAL FRAME (Robot Centric)
   
          ^+Y_global                              ^+Y_robot (Forward)
          |                                       |
          |                                       |
          +-------> +X_global                     +-------> +X_robot (Right/Strafing)
         /                                       /
        v +Z_global (Up)                        v +Z_robot (Up)

   • Orientation Theta (θ): Angle measured counter-clockwise from +X_global to +Y_robot.
   • Wheel 1: Front Right (Quadrant I)   at angle +alpha_f relative to +X_robot
   • Wheel 2: Front Left  (Quadrant II)  at angle +alpha_f relative to -X_robot
   • Wheel 3: Rear Left   (Quadrant III) at angle +alpha_b relative to -X_robot
   • Wheel 4: Rear Right  (Quadrant IV)  at angle +alpha_b relative to +X_robot
==========================================================================
```

The figure after the plant configuration cell draws this layout from the model's own parameters on equal axes, so the angles are true; each arrow is a row of $\mathbf{H}$, the direction along which that wheel pushes the robot for a positive wheel speed.

### ⚙️ Plant Configuration & Initialization

This block configures the physical properties of the system using **estimated not-calibrated constants** and prints the plant report.

* **`COG_FORWARD`, `COG_HEIGHT`**: The centre of mass position forward of the base centre (centred for now; the CAD value is 10.2 mm) and its height (the CAD value, 61.1 mm), set independently.
* **Named constants**: The discrete time step, chassis mass and inertia, wheel geometry, motor constants (Pololu 37D 30:1), the supply voltage limit, and floor friction parameters. Robot geometry, mass, CoG and floor friction come from `robot_constants.py`.
* **`ROLLER_FRICTION`**: The on/off switch for the roller (a-axis) friction. Off by default.

In [ ]:
"""
Constants
=========

Everything here is fixed for a run. The [CHANGE n] tags point to the entries in
dana-models/open_loop_model_changes.md.

Sources:
    robot_constants.py    robot geometry, mass, inertia, CoG and floor friction
    Dana's v3             Pololu 37D 30:1 gearmotor and the supply limit
    chosen, not measured  LuGre and roller friction parameters
"""

import numpy as np

# Robot size and mass: mass [kg], gravity [m/s^2], base circle radius [m], wheel radius [m], wheel angles [deg]
from robot_constants import ROBOT_MASS, GRAVITY, WHEEL_ROBOT_DIST, WHEEL_RADIUS, WHEEL_ALPHA_DEG
# CAD values: yaw inertia about the CoG [kg mm^2], CoG height [mm]
from robot_constants import IZZ_COG_KG_MM2, COG_Z_MM
# Drive friction: static and sliding coefficients, their decay with slip [s/m], rolling resistance coefficient
from robot_constants import MU_D_STAT, MU_D_DYN, ALPHA_D as MU_D_DECAY, WHEEL_ROLLING_RESISTANCE_COEFF
# Roller friction: coefficients at zero and high slip, their decay with slip [s/m], slope of the smoothed sign [s/m]
from robot_constants import MU_A_STAT, MU_A_DYN, ALPHA_A as MU_A_DECAY, K_SMOOTH_A
# Slip above which a wheel is reported as slipping [m/s], speed that smooths the rolling drag at zero [rad/s]
from robot_constants import DRIVE_BREAKAWAY_BAND, WHEEL_LOSS_SMOOTHING_SPEED

# ---------------------------------------------------------------------------
# Time step
# ---------------------------------------------------------------------------

DT = 0.001                       # [s] plant step, 1 kHz

# ---------------------------------------------------------------------------
# Chassis [CHANGE 0, 6]
# ---------------------------------------------------------------------------

ROBOT_INERTIA = IZZ_COG_KG_MM2 * 1e-6    # [kg m^2] CAD yaw inertia about the CoG, wherever the CoG sits
COG_FORWARD = 0.0                        # [m] CoG forward of the base centre. Centred for now; CAD value: 0.0102
COG_HEIGHT = COG_Z_MM * 1e-3             # [m] CoG height above the floor. Set independently of COG_FORWARD

# Dynamics are measured about the CoG, so the mass matrix is diagonal
MASS_MATRIX = np.diag([ROBOT_MASS, ROBOT_MASS, ROBOT_INERTIA])
MASS_MATRIX_INV = np.linalg.inv(MASS_MATRIX)

# ---------------------------------------------------------------------------
# Wheel layout [CHANGE 0]
# ---------------------------------------------------------------------------

FRONT_ANGLE = np.radians(WHEEL_ALPHA_DEG[0])            # [rad] 30 deg, front wheels above +x (right) (alpha_f)
REAR_ANGLE = np.radians(360.0 - WHEEL_ALPHA_DEG[3])     # [rad] 45 deg, rear wheels below +x (right) (alpha_b)

# ---------------------------------------------------------------------------
# Motor: Pololu 37D 30:1, values at the wheel, from Dana's v3 [CHANGE 4, 8]
# ---------------------------------------------------------------------------

V_MAX = 12.0                     # [V] largest voltage the supply can apply
MOTOR_RESISTANCE = 2.18          # [ohm] armature resistance (R)
MOTOR_INDUCTANCE = 0.0023        # [H] armature inductance (L_m)
MOTOR_KT = 0.255                 # [N m/A] torque constant
MOTOR_KB = 0.255                 # [V/(rad/s)] back-EMF constant
WHEEL_INERTIA = 0.00018          # [kg m^2] reflected rotor assembly inertia (J_w)
RIM_MASS = WHEEL_INERTIA / WHEEL_RADIUS**2    # [kg] J_w / r^2, what the LuGre bristles act against

# ---------------------------------------------------------------------------
# Floor friction, drive (d) axis: LuGre [CHANGE 7]
# ---------------------------------------------------------------------------

LUGRE_DEFLECTION = 2e-6          # [m] bristle deflection at the static limit mu_s * F_n
LUGRE_DAMPING_RATIO = 1.0        # fraction of critical damping, against the rim mass
LUGRE_DAMPING_SLIP = 5e-3        # [m/s] slip speed over which the bristle damping fades out
LUGRE_VISCOUS = 0.0              # [N s/m] viscous coefficient sigma_2
LUGRE_CREEP = 1e-4               # [m/s] creep slip, smooths |slip| at zero

# ---------------------------------------------------------------------------
# Wheel solve: backward Euler [CHANGE 5]
# ---------------------------------------------------------------------------

IMPLICIT_TOL = 1e-4              # [m/s] repeat the solve until the slip changes by less than this
IMPLICIT_MAX_ITER = 50           # most solves per step

# ---------------------------------------------------------------------------
# Roller friction, axle (a) axis [CHANGE 9]
# ---------------------------------------------------------------------------

ROLLER_FRICTION = False          # on/off switch. False = free rollers, as in Dana's model

# ---------------------------------------------------------------------------
# Disturbances
# ---------------------------------------------------------------------------

SIGMA_LOAD = 0.15                # [N] standard deviation of the structural vibration on the wheel loads (sigma_V in the spec)
SIGMA_MU = 0.03                  # standard deviation of the floor friction variation
RANDOM_SEED = 42                 # the same seed every run, so runs are repeatable

In [ ]:
# Plant report: the constants in use
def print_plant_report(geometry):
    """Print the constants in use.

    geometry: from wheel_geometry
    """
    line = "-" * 71

    print("=" * 71)
    print("             OMNI-WHEEL PHYSICAL PLANT REPORT")
    print("=" * 71)
    print(f" Sampling rate {1.0 / DT:.0f} Hz, DT = {DT} s")
    print(line)
    print(" CHASSIS")
    print(f"  Mass:                     {ROBOT_MASS:.3f} kg")
    print(f"  CoG height / forward:     {COG_HEIGHT:.4f} m / {COG_FORWARD:.4f} m")
    print(f"  Yaw inertia about CoG:    {ROBOT_INERTIA:.5f} kg m^2")
    print(f"  Base radius:              {WHEEL_ROBOT_DIST:.4f} m")
    print(f"  Wheel angles front/rear:  {np.degrees(FRONT_ANGLE):.1f} / {np.degrees(REAR_ANGLE):.1f} deg")
    print(line)
    print(" MOTOR AND WHEEL")
    print(f"  Supply limit:             {V_MAX:.1f} V")
    print(f"  R / L:                    {MOTOR_RESISTANCE:.3f} ohm / {MOTOR_INDUCTANCE:.6f} H")
    print(f"  Kt / Kb:                  {MOTOR_KT:.4f} N m/A / {MOTOR_KB:.4f} V/(rad/s)")
    print(f"  Wheel inertia / radius:   {WHEEL_INERTIA:.7f} kg m^2 / {WHEEL_RADIUS:.4f} m")
    print(line)
    print(" FRICTION AND DISTURBANCES")
    print(f"  Rolling resistance:       {WHEEL_ROLLING_RESISTANCE_COEFF:.4f}")
    print(f"  Drive mu static/sliding:  {MU_D_STAT:.3f} / {MU_D_DYN:.3f}")
    print(f"  Roller friction:          {'on' if ROLLER_FRICTION else 'off'}, mu {MU_A_STAT:.3f} -> {MU_A_DYN:.3f}")
    print(f"  Noise on loads / mu:      {SIGMA_LOAD:.3f} N / {SIGMA_MU:.3f}")
    print(line)
    print(f" H:\n{geometry['H']}\n")
    print(f" Mass matrix:\n{MASS_MATRIX}")
    print("=" * 71)


print_plant_report(wheel_geometry(FRONT_ANGLE, REAR_ANGLE))

In [ ]:
# [CHANGE 3] New cell, not in Dana's original or her v3: draws the wheel layout from the model's own parameters.
# Spatial reference frames and wheel layout, drawn from the model's own parameters (equal axes, so the angles are true)
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Arc


def polar(radius, degrees):
    """The (x, y) point at this radius and angle.

    radius: distance from the origin
    degrees: angle, counter-clockwise from +x [deg]
    """
    return radius * np.cos(np.radians(degrees)), radius * np.sin(np.radians(degrees))


def arrow(ax, tip, tail, **style):
    """Draw an arrow on these axes.

    ax: the axes to draw on
    tip, tail: (x, y) of the two ends
    style: matplotlib arrow properties
    """
    ax.annotate("", xy=tip, xytext=tail, arrowprops=dict(**style))


def draw_layout(front_angle, rear_angle):
    """Draw the global frame and the robot's wheel layout, side by side.

    front_angle: angle of the front wheels above the x axis (alpha_f) [rad]
    rear_angle: angle of the rear wheels below the x axis (alpha_b) [rad]
    """
    geometry = wheel_geometry(front_angle, rear_angle)
    angles = wheel_angles(front_angle, rear_angle)
    radius_mm = WHEEL_ROBOT_DIST * 1e3
    wheel_xy = radius_mm * np.column_stack((np.cos(angles), np.sin(angles)))
    drive = geometry["H"][:, :2]

    fig, (ax_global, ax) = plt.subplots(1, 2, figsize=(14, 6.5), gridspec_kw={"width_ratios": [1, 2]})

    # Global frame: X right, Y up; heading theta positive CCW
    arrow(ax_global, (1.0, 0), (0, 0), arrowstyle="->", lw=2)
    arrow(ax_global, (0, 1.0), (0, 0), arrowstyle="->", lw=2)
    ax_global.text(1.05, 0, "+X_global", va="center")
    ax_global.text(0, 1.05, "+Y_global", ha="center")
    ax_global.add_patch(Arc((0, 0), 0.9, 0.9, theta1=0, theta2=60, color="C3", lw=1.5))
    arrow(ax_global, polar(0.45, 62), polar(0.45, 55), arrowstyle="->", color="C3", lw=1.5)
    ax_global.text(0.5, 0.25, "heading theta:\nCCW positive", color="C3")
    ax_global.text(-0.15, -0.3, "At theta = 0 the robot's +Y_robot (forward)\npoints along +Y_global, and +X_robot along +X_global.", fontsize=9)
    ax_global.set(xlim=(-0.2, 1.4), ylim=(-0.45, 1.3), title="Global frame (fixed room)")
    ax_global.set_aspect("equal")
    ax_global.axis("off")

    # Local frame: base circle and axes
    limit = 1.35 * radius_mm
    ax.add_patch(plt.Circle((0, 0), radius_mm, fill=False, ls=":", color="0.6"))
    arrow(ax, (limit, 0), (-limit, 0), arrowstyle="->", color="0.3")
    arrow(ax, (0, limit), (0, -limit), arrowstyle="->", color="0.3")
    ax.text(limit, -6, "+X_robot (right)", ha="right", va="top")
    ax.text(4, limit, "+Y_robot (forward)", va="top")

    # Local frame: wheel positions and drive directions
    names = ("1 FR", "2 FL", "3 RL", "4 RR")
    for i in range(4):
        px, py = wheel_xy[i]
        ax.plot([0, px], [0, py], color=f"C{i}", lw=1)
        ax.plot(px, py, "o", color=f"C{i}", ms=9)
        arrow(ax, (px + 25 * drive[i, 0], py + 25 * drive[i, 1]), (px, py),
              arrowstyle="-|>", color=f"C{i}", lw=2)
        ax.text(px * 1.17, py * 1.17, f"wheel {names[i]}\n{np.degrees(angles[i]) % 360:.0f} deg",
                      ha="center", va="center", color=f"C{i}", fontsize=9)

    # Front and rear wheel angles, measured from the +X_robot axis and labelled with the spec's symbols
    front_deg, rear_deg = np.degrees(front_angle), np.degrees(rear_angle)
    ax.add_patch(Arc((0, 0), 50, 50, theta1=0, theta2=front_deg, color="C0", lw=1.5))
    ax.text(*polar(30, front_deg / 2), f"alpha_f = {front_deg:.0f} deg", color="C0", fontsize=9)
    ax.add_patch(Arc((0, 0), 70, 70, theta1=-rear_deg, theta2=0, color="C3", lw=1.5))
    ax.text(*polar(40, -rear_deg / 2), f"alpha_b = {rear_deg:.0f} deg", color="C3", fontsize=9)

    ax.set(xlim=(-limit, limit), ylim=(-limit, limit), xlabel="X_robot [mm]", ylabel="Y_robot [mm]",
                 title="Local frame: wheel positions (dots) and drive directions (arrows, rows of H)")
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.2)

    plt.tight_layout()
    plt.show()


draw_layout(FRONT_ANGLE, REAR_ANGLE)

# 🧪 Kinematic & Dynamic Verification Testing

This section executes open-loop verification trials to validate the electromechanical motor re-coupling, asymmetric kinematics mapping ($\mathbf{H}$), and stick-slip friction boundaries under hardware-verified sign directions. 

### 📐 Structural Testing Grid Architecture
The execution pipeline performs a **multi-block comparative analysis** across two distinct physical frame designs and environmental settings:
1. **Block 1: Symmetric Frame Verification:** Configures the plant layout to an identical 45° alignment across all wheel lines ($\alpha_f = 45^\circ$, $\alpha_b = 45^\circ$).
2. **Block 2: Asymmetric Frame Verification:** Reverts the plant setup back to your custom asymmetric layout ($\alpha_f = 30^\circ$, $\alpha_b = 45^\circ$).
3. **Block 3: Asymmetric Frame With Boosted Voltage Verification:** Keeps the asymmetric frame but boosts the voltage to operate the voltage is a more ideal range but also cause slipping. The previous two verifications focus on the voltage direction and general kinematic so voltage is very low.   

### 🕹️ Verification Test Input Sequences
To isolate each independent degree of freedom, the simulation applies precise step voltage patterns across the four wheel motors:
*   **Test 1: Forward Motion ($\mathbf{V} = [V, -V, -V, V]^\top$):** Applies positive voltages to the right-side motors ($1, 4$) and negative voltages to the left-side motors ($2, 3$). This drives all wheels to push the ground backward symmetrically, canceling out lateral side forces and accelerating the robot straight ahead along its local $+Y_r$ axis.
*   **Test 2: Lateral Side-Strafing ($\mathbf{V} = [-V, -V, V, V]^\top$):** Applies negative voltages to the front axle motors ($1, 2$) and positive voltages to the rear axle motors ($3, 4$). This mirrors the longitudinal forces across the front-to-back split line, causing forward and backward vectors to cancel out. The remaining lateral components combine to push the chassis cleanly sideways to the right along its local $+X_r$ axis.
*   **Test 3: Pure Rotational Spinning ($\mathbf{V} = [V, V, V, V]^\top$):** Applies uniform positive voltages to all four wheels simultaneously. Because every omni-wheel is mounted tangentially to the chassis base circle, spinning them all in the same rotational direction eliminates all linear translation vectors and creates a pure angular torque, spinning the robot counter-clockwise ($+\theta$) around its Center of Mass.

### 🔄 Dual-Pass State Conditions
To clearly isolate the degradation caused by environmental disturbances, each vehicle geometry executes a sequential dual-pass simulation:
*   **Ideal Pass (Reference Baseline):** Strips all noise configurations ($\sigma_V = 0.0$, $\sigma_\mu = 0.0$). The results render as soft, transparent curves representing perfect theoretical trajectories.
*   **Perturbed Pass (Real-World Overlay):** Activates a repeatable, seeded random number generator stream (`np.random.default_rng`) to overlay high-frequency structural chassis vibrations ($w_F$) and spatial friction coefficient patches ($w_\mu$).

### 📊 Logged Telemetry Variables
Each configuration runs the three directional maneuvers back-to-back over a 0.5-second time window (500 discrete calculation steps) and generates a standalone 2x2 grid capturing:
*   **Applied Motor Terminal Voltages ($V_{1\dots4}$):** Verifies step-input directional polarities.
*   **Wheel Angular Velocities ($\omega_{1\dots4}$):** Monitors electromechanical back-EMF variations and independent rotor spinout integrations ($J_w$).
*   **Chassis Local Frame Speeds ($\dot{x}_r, \dot{y}_r, \dot{\theta}$):** Evaluates translational and angular momentum unbalances inside the mass matrix universe ($\mathbf{M}^{-1}\mathbf{H}^\top\mathbf{F}_k$).
*   **Global Position Tracking ($X-Y$ Trajectory Plane):** Visualizes dead-reckoning coordinate transformation drift paths directly overlaying ideal performance vs. noisy tracking realities.

In [ ]:
"""
Test plots
==========

Shared by every test block below: the three motions and the 2x2 figure comparing a run
with the noise off and on.
"""

import numpy as np
import matplotlib.pyplot as plt

# (test name, wheel voltage signs)
MANEUVERS = (("Test 1: Forward Motion", FORWARD),
             ("Test 2: Lateral Side-Strafing", STRAFE_RIGHT),
             ("Test 3: Pure Rotational Spinning", ROTATE_CCW))

# Line styles: the ideal run is the soft wide line under the noisy run
STYLE_IDEAL = {"alpha": 0.4, "lw": 2.5}
STYLE_NOISY = {"alpha": 1.0, "lw": 1.2}


def finish_axis(ax, title, xlabel, ylabel, **legend):
    """Add the titles, grid and legend to one subplot.

    ax: the subplot
    title, xlabel, ylabel: its labels
    legend: matplotlib legend options
    """
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(True)
    ax.legend(**legend)


def plot_results(ideal, noisy, title, markers=()):
    """2x2 figure: voltages, wheel speeds, local velocities, global path.

    ideal: history of the run with the noise off
    noisy: history of the run with the noise on
    title: figure title
    markers: (step, matplotlib format, label) points to mark on the path
    """
    time = ideal["time"]
    fig, axes = plt.subplots(2, 2, figsize=(15, 11))
    fig.suptitle(title, fontsize=16, fontweight="bold")

    # Applied voltages
    for i in range(4):
        axes[0, 0].plot(time, ideal["V"][:, i], label=f"V{i + 1}")
    finish_axis(axes[0, 0], "Applied Motor Terminal Voltages", "Time (s)", "Voltage (V)", loc="upper right")

    # Wheel speeds
    for i in range(4):
        axes[0, 1].plot(time, ideal["omega"][:, i], color=f"C{i}", label=rf"$\omega_{i + 1}$ Ideal", **STYLE_IDEAL)
        axes[0, 1].plot(time, noisy["omega"][:, i], color=f"C{i}", label=rf"$\omega_{i + 1}$ Noise", **STYLE_NOISY)
    finish_axis(axes[0, 1], "Wheel Angular Velocities Comparison", "Time (s)", "Speed (rad/s)",
                loc="upper right", ncol=2, fontsize="small")

    # Local velocities: x_dot, y_dot, theta_dot
    for j, (color, symbol) in enumerate((("r", r"\dot{x}_r"), ("b", r"\dot{y}_r"), ("g", r"\dot{\theta}"))):
        axes[1, 0].plot(time, ideal["v_local"][:, j], color=color, label=f"${symbol}$ Ideal", **STYLE_IDEAL)
        axes[1, 0].plot(time, noisy["v_local"][:, j], color=color, label=f"${symbol}$ Noise", **STYLE_NOISY)
    finish_axis(axes[1, 0], "Chassis Local Frame Speeds", "Time (s)", "Velocity (m/s or rad/s)",
                loc="upper right", ncol=2, fontsize="small")

    # Global path
    path, ideal_path = noisy["pos_global"], ideal["pos_global"]
    axes[1, 1].plot(ideal_path[:, 0], ideal_path[:, 1], "k--", lw=2.0, label="Ideal Base Path")
    axes[1, 1].plot(path[:, 0], path[:, 1], "b-", lw=1.5, label="Perturbed Path")
    axes[1, 1].plot(path[0, 0], path[0, 1], "go", markersize=8, label="Start")
    for k, fmt, label in markers:
        axes[1, 1].plot(path[k, 0], path[k, 1], fmt, markersize=8, label=label)
    axes[1, 1].plot(path[-1, 0], path[-1, 1], "rx", markersize=8, label="End")
    axes[1, 1].set_aspect("equal", "datalim")
    finish_axis(axes[1, 1], "Global Spatial Position Tracking (X-Y)", "Global X Position (m)", "Global Y Position (m)", loc="best")

    plt.tight_layout()
    plt.show()

In [ ]:
# Block 1: symmetric 45-45 layout
geometry = wheel_geometry(np.radians(45.0), np.radians(45.0))
low_voltage = 1.0                  # [V] low, to check the voltage directions and the kinematics
steps = int(round(0.5 / DT))       # 0.5 s run

print("Executing Block 1: Symmetric 45-45 Wheel Angle Configurations...")

for name, pattern in MANEUVERS:
    ideal, noisy = run_ideal_and_noisy(constant_voltage(low_voltage, pattern), steps, geometry)
    plot_results(ideal, noisy, f"Symmetric Block (45-45 Alignment) | {name}")

In [ ]:
# Block 2: our asymmetric 30-45 layout
geometry = wheel_geometry(FRONT_ANGLE, REAR_ANGLE)
low_voltage = 1.0                  # [V] low, to check the voltage directions and the kinematics
steps = int(round(0.5 / DT))       # 0.5 s run

print("Executing Block 2: Asymmetric 30-45 Wheel Angle Configurations...")

for name, pattern in MANEUVERS:
    ideal, noisy = run_ideal_and_noisy(constant_voltage(low_voltage, pattern), steps, geometry)
    plot_results(ideal, noisy, f"Asymmetric Block (30-45 Alignment) | {name}")

In [ ]:
# Block 3: 30-45 at a boosted voltage, enough to make the wheels slip
geometry = wheel_geometry(FRONT_ANGLE, REAR_ANGLE)
boost_voltage = 10.0               # [V]
steps = int(round(0.5 / DT))       # 0.5 s run

print(f"Executing Block 3: Asymmetric 30-45 Wheel Angle Configurations at High Voltage ({boost_voltage:.1f}V)...")

for name, pattern in MANEUVERS:
    ideal, noisy = run_ideal_and_noisy(constant_voltage(boost_voltage, pattern), steps, geometry)
    plot_results(ideal, noisy, f"Asymmetric High-Volt Block (30-45 @ {boost_voltage:.1f}V) | {name} (High Volt)")

# 🧪 Multi-Stage Sequence Verification Testing

This section executes verification routines using a **Multi-Stage Step Profile Timeline**. Each directional maneuver commands a voltage profile divided into three 1.0-second stages (Positive Drive $\to$ Negative Plug-Braking $\to$ Zero Coasting). This explicitly evaluates the plant's transient response, deceleration dynamics, and LuGre continuous friction behavior without artificial velocity snapping or numerical chatter.

In [ ]:
# Multi-stage: 30-45, drive, reverse, then coast
geometry = wheel_geometry(FRONT_ANGLE, REAR_ANGLE)
stage_voltage = 4.0                        # [V]
steps_per_stage = int(round(1.0 / DT))     # 1.0 s per stage
steps = 3 * steps_per_stage

# (test name, wheel voltage signs). Dana's side-to-side strafes left first
stage_maneuvers = (("Forward/Backward", FORWARD), ("Side-to-Side", -STRAFE_RIGHT), ("CCW/CW Rotation", ROTATE_CCW))
# Where the voltage switches: (step, matplotlib format, label)
markers = ((steps_per_stage, "y^", "Switch to Neg"), (2 * steps_per_stage, "mo", "Switch to Zero"))

print("Executing Profile Tests under Asymmetric 30-45 Geometry...")

for name, pattern in stage_maneuvers:
    controller = multi_stage_voltage(stage_voltage, pattern, steps_per_stage)
    ideal, noisy = run_ideal_and_noisy(controller, steps, geometry)
    plot_results(ideal, noisy, f"Asymmetric Chassis (30-45) | {name}: Pos/Neg/Zero Sequence", markers)